# Demo Campaign Seed

Three-task job for seeding the dev environment with real data:
1. **grant_demo_access** (deployer): grants app SPN read on wanderBricks Git folder,
   UC + warehouse access to M2M SPN for OBO import.
2. **seed** (M2M OAuth): registers source, imports metric views + glossary terms,
   creates a demo campaign, and casts votes.
3. **verify** (M2M OAuth): `/api/leaderboard` non-empty, campaign asset count > 0.

Controlled by the `task` widget. Idempotent.

In [0]:
%pip install --upgrade databricks-sdk requests
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("task", "seed", "Task to run")
dbutils.widgets.text("app_name", "dev-semantic-ground-truth", "App name")
dbutils.widgets.text("scope_name", "semantic_ground_truth_credentials", "Secret scope")
dbutils.widgets.text("client_id_key", "m2m_client_id_dev", "M2M client_id key")
dbutils.widgets.text("client_secret_key", "m2m_client_secret_dev", "M2M client_secret key")
dbutils.widgets.text("catalog", "hls_fde_dev", "UC catalog")
dbutils.widgets.text("schema_name", "dev_matthew_giglia_semantic_ground_truth", "UC schema")
dbutils.widgets.text("warehouse_id", "0ea84986a23a47c3", "SQL warehouse")
dbutils.widgets.text("git_folder_path",
    "/Workspace/Users/matthew.giglia@databricks.com/wanderBricksSemantics-care",
    "WanderBricks Git folder")
dbutils.widgets.text("metric_view_schema",
    "hls_fde_dev.dev_matthew_giglia_wb_metric_views_care",
    "Schema with source metric views")

task = dbutils.widgets.get("task")
app_name = dbutils.widgets.get("app_name")
scope_name = dbutils.widgets.get("scope_name")
client_id_key = dbutils.widgets.get("client_id_key")
client_secret_key = dbutils.widgets.get("client_secret_key")
catalog = dbutils.widgets.get("catalog")
schema_name = dbutils.widgets.get("schema_name")
warehouse_id = dbutils.widgets.get("warehouse_id")
git_folder_path = dbutils.widgets.get("git_folder_path")
metric_view_schema = dbutils.widgets.get("metric_view_schema")

assert task in ("grant_demo_access", "seed", "verify"), f"Unknown task: {task}"
print(f"Task: {task}")
print(f"App:  {app_name}")

In [0]:
import os, json
import requests as req
from databricks.sdk import WorkspaceClient

# Job-identity client (grant_demo_access uses this)
w = WorkspaceClient()

# App URL + SPN discovery
_app = w.apps.get(app_name)
_raw_url = _app.url or ""
app_url = (_raw_url if _raw_url.startswith("https://") else f"https://{_raw_url}").rstrip("/")
app_spn_client_id = _app.service_principal_client_id
print(f"App URL: {app_url}")
print(f"App SPN client_id: {app_spn_client_id}")


def m2m_session():
    """Build an authenticated requests.Session using M2M OAuth."""
    cid = dbutils.secrets.get(scope=scope_name, key=client_id_key)
    csecret = dbutils.secrets.get(scope=scope_name, key=client_secret_key)
    host = os.environ.get("DATABRICKS_HOST", "https://fevm-hls-fde.cloud.databricks.com")
    m2m_w = WorkspaceClient(host=host, client_id=cid, client_secret=csecret)
    s = req.Session()
    s.headers.update(m2m_w.config.authenticate())
    s.max_redirects = 0
    return s, cid

In [0]:
def grant_demo_access():
    """Grant minimum access for demo data import (deployer identity)."""
    from databricks.sdk.service.iam import AccessControlRequest, PermissionLevel
    from databricks.sdk.service import catalog as c

    # 1. Resolve app SPN
    spn = None
    for sp in w.service_principals.list(filter=f'applicationId eq "{app_spn_client_id}"'):
        spn = sp
        break
    assert spn, f"App SPN not found for client_id={app_spn_client_id}"
    print(f"App SPN: {spn.display_name} (id={spn.id})")

    # 2. CAN_READ on wanderBricks Git folder -> app SPN
    info = w.workspace.get_status(git_folder_path)
    w.permissions.update(
        request_object_type="repos",
        request_object_id=str(info.object_id),
        access_control_list=[
            AccessControlRequest(
                service_principal_name=spn.application_id,
                permission_level=PermissionLevel.CAN_READ,
            )
        ],
    )
    print(f"\u2713 App SPN CAN_READ on {git_folder_path}")

    # 3. Resolve M2M SPN for UC grants
    m2m_client_id = dbutils.secrets.get(scope=scope_name, key=client_id_key)
    m2m_spn = None
    for sp in w.service_principals.list(filter=f'applicationId eq "{m2m_client_id}"'):
        m2m_spn = sp
        break
    assert m2m_spn, f"M2M SPN not found for client_id (from scope)"
    print(f"M2M SPN: {m2m_spn.display_name} (id={m2m_spn.id})")

    # 4. UC grants on metric view schema -> M2M SPN
    mv_catalog = metric_view_schema.split(".")[0]
    w.grants.update(
        securable_type="catalog", full_name=mv_catalog,
        changes=[c.PermissionsChange(principal=m2m_spn.application_id,
                                     add=[c.Privilege.USE_CATALOG])],
    )
    w.grants.update(
        securable_type="schema", full_name=metric_view_schema,
        changes=[c.PermissionsChange(principal=m2m_spn.application_id,
                                     add=[c.Privilege.USE_SCHEMA, c.Privilege.SELECT])],
    )
    print(f"\u2713 M2M SPN USE CATALOG + USE SCHEMA + SELECT on {metric_view_schema}")

    # 5. Warehouse CAN_USE -> M2M SPN
    w.warehouses.update_permissions(
        warehouse_id=warehouse_id,
        access_control_list=[
            AccessControlRequest(
                service_principal_name=m2m_spn.application_id,
                permission_level=PermissionLevel.CAN_USE,
            )
        ],
    )
    print(f"\u2713 M2M SPN CAN_USE on warehouse {warehouse_id}")

    return {"grants": "applied"}

In [0]:
def seed():
    """Register source, import assets, create campaign, vote (M2M OAuth)."""
    s, _m2m_id = m2m_session()

    CAMPAIGN_NAME = "Demo \u2014 WanderBricks semantics"
    METRIC_VIEWS = [
        f"{metric_view_schema}.mv_bookings",
        f"{metric_view_schema}.mv_reviews",
        f"{metric_view_schema}.mv_payments",
    ]

    # 1. Register Git folder source (idempotent)
    r = s.get(f"{app_url}/api/sources", timeout=30, allow_redirects=False)
    r.raise_for_status()
    sources = r.json()
    source = next((src for src in sources if src.get("workspacePath") == git_folder_path), None)
    if not source:
        r = s.post(
            f"{app_url}/api/sources",
            json={"workspacePath": git_folder_path, "displayName": "WanderBricks Semantics"},
            timeout=30, allow_redirects=False,
        )
        r.raise_for_status()
        source = r.json()
        print(f"\u2713 Registered source: {source.get('id')}")
    else:
        print(f"\u2713 Source already registered: {source.get('id')}")
    source_id = source["id"]

    # 2. Scan
    r = s.post(f"{app_url}/api/sources/{source_id}/scan", timeout=60, allow_redirects=False)
    r.raise_for_status()
    scan = r.json()
    print(f"\u2713 Scan: {len(scan.get('metricViews', []))} metric views, "
          f"{len(scan.get('glossaryTerms', []))} glossary terms")

    # 3. Import
    import_payload = {"metricViews": METRIC_VIEWS, "sourceId": source_id}
    glossary_terms = scan.get("glossaryTerms", [])
    if glossary_terms:
        import_payload["glossaryTerms"] = [
            {"sourceId": source_id, "filePath": t["filePath"], "term": t["term"]}
            for t in glossary_terms
        ]
    r = s.post(f"{app_url}/api/assets/import", json=import_payload, timeout=60, allow_redirects=False)
    r.raise_for_status()
    imported = r.json()
    asset_ids = imported.get("assetIds", [])
    print(f"\u2713 Imported {len(asset_ids)} assets")

    # 4. Create campaign if absent
    r = s.get(f"{app_url}/api/campaigns", timeout=30, allow_redirects=False)
    r.raise_for_status()
    campaigns = r.json()
    campaign = next((c for c in campaigns if c.get("name") == CAMPAIGN_NAME), None)
    created_campaign = not campaign
    if not campaign:
        r = s.post(
            f"{app_url}/api/campaigns",
            json={
                "name": CAMPAIGN_NAME,
                "campaignType": "manual",
                "scopeAssetIds": asset_ids,
                "rbacGroups": ["semantic-ground-truth-users"],
            },
            timeout=30, allow_redirects=False,
        )
        r.raise_for_status()
        campaign = r.json()
        print(f"\u2713 Created campaign: {campaign.get('id')}")
    else:
        print(f"\u2713 Campaign exists: {campaign.get('id')}")
    campaign_id = campaign["id"]

    # 5. Vote on review cards (mix of approve/reject/edit)
    vote_types = ["approve", "reject", "approve", "approve", "edit",
                  "approve", "reject", "approve", "approve", "approve"]
    votes_cast = 0
    for i in range(10):
        r = s.get(
            f"{app_url}/api/review/next",
            params={"campaignId": campaign_id},
            timeout=30, allow_redirects=False,
        )
        r.raise_for_status()
        # Contract: GetNextResponse = ReviewCard | null (JSON null when the queue is empty)
        card = r.json() if r.content else None
        if not card:
            print(f"  No more cards to review (cast {votes_cast})")
            break
        asset_id = card["asset"]["id"]
        vt = vote_types[i % len(vote_types)]
        vote_body = {"assetId": asset_id, "voteType": vt}
        if vt == "edit":
            vote_body["feedback"] = "Suggested edit: clarify the description."
        r = s.post(f"{app_url}/api/review/vote", json=vote_body, timeout=30, allow_redirects=False)
        r.raise_for_status()
        votes_cast += 1

    print(f"\u2713 Cast {votes_cast} votes")
    # A newly created campaign must yield \u22653 votes; reruns may find the queue already reviewed
    assert votes_cast >= 3 or not created_campaign, "First run must cast at least 3 votes"
    return {"source_id": source_id, "campaign_id": campaign_id, "votes": votes_cast, "assets": len(asset_ids)}

In [0]:
def verify():
    """Check leaderboard + campaign asset count (M2M OAuth)."""
    s, _ = m2m_session()

    # 1. Leaderboard non-empty
    r = s.get(f"{app_url}/api/leaderboard", timeout=30, allow_redirects=False)
    r.raise_for_status()
    leaderboard = r.json()
    assert len(leaderboard) > 0, "Leaderboard is empty \u2014 no votes recorded"
    print(f"\u2713 Leaderboard: {len(leaderboard)} entries")

    # 2. Campaign has assets
    r = s.get(f"{app_url}/api/campaigns", timeout=30, allow_redirects=False)
    r.raise_for_status()
    campaigns = r.json()
    demo = next((c for c in campaigns if "WanderBricks" in c.get("name", "")), None)
    assert demo, "Demo campaign not found"
    asset_count = demo.get("totalAssets", 0)
    assert asset_count > 0, "Campaign has 0 assets"
    print(f"\u2713 Campaign '{demo['name']}': {asset_count} assets")

    return {"leaderboard_entries": len(leaderboard), "campaign_assets": asset_count}

In [0]:
TASKS = {
    "grant_demo_access": grant_demo_access,
    "seed": seed,
    "verify": verify,
}
fn = TASKS[task]
print(f"\n=== Running task: {task} ===\n")
result = fn()
print(f"\n=== Task {task} complete ===")
dbutils.notebook.exit(json.dumps({"task": task, "result": result}))